# 03 · Attention audit: Grad-CAM vs expert lesion masks

Post-hoc check of where the reference ResNet-50 (seed 42) looks on the test set, measured against the HAM10000 lesion segmentations (Tschandl et al., 2019; Kaggle `tschandl/ham10000-lesion-segmentations`).

- Grad-CAM on `layer4[-1].conv3`, target = predicted class, upsampled to 224 × 224.
- Masks resized to the same grid (nearest). Validation/test preprocessing is a plain resize, so both are pixel-aligned.
- Metrics per image: energy proportion (EP, threshold-free), pointing (peak inside mask), IoU at 0.5 on the min-max normalised map.
- Only test images with a mask are audited (1991 / 2348). The images added from the official ISIC 2018 val/test partitions have no mask.

Runs on Colab. Expects in `DRIVE_ROOT`: the images, the three ground-truth CSVs, `colab/split_assignment.json`, `colab/resnet50_best.pth` and the mask folder `HAM10000_segmentations_lesion_tschandl/`.

Outputs (in `OUT_DIR`): `gradcam_segmentation_overlap.json`, `fig_gradcam_seg_energy.pdf`, `fig_gradcam_seg_panel.pdf`, `fig_gradcam_seg_errors.pdf`. Committed copy of the JSON: `results/attention/gradcam_segmentation_overlap.json`.

Cell 4 only needs the JSON: pooled and per-class Mann-Whitney tests and the lesion-size check run without GPU or data.

In [ ]:
# 1 · Setup: locate split, labels, images, checkpoint and masks; end-to-end check on one image.
# GPU optional here. Stops with an explicit message if anything is missing.

import os, glob, json, csv
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision.models import resnet50
import albumentations as A
from albumentations.pytorch import ToTensorV2

DRIVE_ROOT = "/content/drive/MyDrive/ISIC2018_Task3"
COLAB_DIR = os.path.join(DRIVE_ROOT, "colab")        # reference run (read only)
MASKS_DIR = os.path.join(DRIVE_ROOT, "HAM10000_segmentations_lesion_tschandl")
OUT_DIR = COLAB_DIR

GRADCAM_TARGET = 'pred'     # 'pred': predicted class | 'true': ground-truth class
COMPARE_SIZE = 224
IOU_THR = 0.5

CLASSES = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
MALIGNANT = {'MEL', 'BCC', 'AKIEC'}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def _mount_drive():
    if not os.path.isdir("/content/drive"):
        from google.colab import drive
        drive.mount('/content/drive')
    assert os.path.isdir(DRIVE_ROOT), f"{DRIVE_ROOT} not found"


def _find_one(filename, root, required=True):
    hits = glob.glob(os.path.join(root, "**", filename), recursive=True)
    if not hits:
        if required:
            raise FileNotFoundError(f"'{filename}' not found under {root}")
        return None
    if len(hits) > 1:
        print(f"  [warn] several matches for '{filename}', using the first:")
        for h in hits:
            print(f"    {h}")
    return hits[0]


def _load_label_map():
    # the three official CSVs together cover every image; the split comes from split_assignment.json
    csv_names = ["ISIC2018_Task3_Training_GroundTruth.csv",
                 "ISIC2018_Task3_Validation_GroundTruth.csv",
                 "ISIC2018_Task3_Test_GroundTruth.csv"]
    label_map = {}
    for name in csv_names:
        path = _find_one(name, DRIVE_ROOT, required=False)
        if path is None:
            print(f"  [warn] {name} not found, skipping")
            continue
        with open(path, newline='') as f:
            for row in csv.DictReader(f):
                onehot = {c: float(row[c]) for c in CLASSES}
                label_map[row['image']] = max(onehot, key=onehot.get)
        print(f"  [ok] {name}")
    assert label_map, "no ground-truth CSV found"
    return label_map


def _index_raw_images():
    # same rule as index_images() in the main pipeline: first occurrence per id, sorted
    per_id = {}
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        for path in glob.glob(os.path.join(DRIVE_ROOT, "**", ext), recursive=True):
            img_id = os.path.splitext(os.path.basename(path))[0]
            if img_id not in per_id:
                per_id[img_id] = path
    return dict(sorted(per_id.items()))


def _build_mask_index(masks_dir):
    idx = {}
    for p in glob.glob(os.path.join(masks_dir, "**", "*.png"), recursive=True):
        base = os.path.splitext(os.path.basename(p))[0]
        for suf in ("_segmentation", "_Segmentation", "_seg", "_mask", "_Mask"):
            if base.endswith(suf):
                base = base[: -len(suf)]
                break
        idx[base] = p
    return idx


def load_reference_resnet50(ckpt_path, num_classes=len(CLASSES)):
    m = resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    sd = state['state_dict'] if isinstance(state, dict) and 'state_dict' in state else state
    m.load_state_dict(sd)
    return m.to(device).eval()


def compute_gradcam(model, image_tensor, target_class):
    tl = model.layer4[-1].conv3
    grads, acts = [], []
    def _sg(g): grads.append(g.detach().cpu())
    def _fh(mod, i, o): acts.append(o.detach().cpu()); o.register_hook(_sg)
    handle = tl.register_forward_hook(_fh)
    model.zero_grad()
    out = model(image_tensor.unsqueeze(0).to(device))
    pred = int(out.argmax(1).item())
    tgt = pred if target_class is None else target_class
    out[0, tgt].backward()
    handle.remove()
    g = grads[0].squeeze().numpy(); a = acts[0].squeeze().numpy()
    cam = np.maximum(np.einsum('c,chw->hw', g.mean(axis=(1, 2)), a), 0)
    mn, mx = cam.min(), cam.max()
    if mx - mn > 1e-8:
        cam = (cam - mn) / (mx - mn)
    return cam, pred


def upsample_cam(cam, size):
    im = Image.fromarray((cam * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    return np.array(im).astype(np.float32) / 255.0


def load_mask(mask_path, size):
    m = np.array(Image.open(mask_path).convert('L').resize((size, size), Image.NEAREST))
    return m > 127


print("1) Drive")
_mount_drive()

print("2) split_assignment.json")
assert os.path.isdir(COLAB_DIR), f"{COLAB_DIR} not found"
SPLIT_PATH = _find_one("split_assignment.json", COLAB_DIR)
with open(SPLIT_PATH) as f:
    split_assignment = json.load(f)
test_ids_all = sorted(iid for iid, sp in split_assignment.items() if sp == 'test')   # sorted: set order changes between sessions
print(f"  [ok] {len(test_ids_all)} test ids")

print("3) labels")
label_map = _load_label_map()

print("4) images")
id2path = _index_raw_images()
test_records_raw = [(iid, id2path[iid], label_map[iid])
                    for iid in test_ids_all if iid in id2path and iid in label_map]
missing_path = sum(1 for iid in test_ids_all if iid not in id2path)
missing_label = sum(1 for iid in test_ids_all if iid in id2path and iid not in label_map)
print(f"  [ok] {len(id2path)} indexed, {len(test_records_raw)} test images resolved")
if missing_path:
    print(f"  [warn] {missing_path} test ids without image file")
if missing_label:
    print(f"  [warn] {missing_label} test ids without label")
assert test_records_raw, "no test image resolved"

print("5) checkpoint")
try:
    RESNET50_CKPT = _find_one("resnet50_best.pth", COLAB_DIR)
except FileNotFoundError:
    RESNET50_CKPT = _find_one("resnet50_best.pth", DRIVE_ROOT)
print(f"  [ok] {RESNET50_CKPT}")

print("6) masks")
assert os.path.isdir(MASKS_DIR), f"{MASKS_DIR} not found"
mask_index = _build_mask_index(MASKS_DIR)
print(f"  [ok] {len(mask_index)} masks")

print("7) mask coverage on test")
per_class_records = {c: [] for c in CLASSES}
for iid, path, cls in test_records_raw:
    per_class_records[cls].append((iid, path))
total_test = total_hit = 0
for c in CLASSES:
    recs = per_class_records[c]
    hit = sum(1 for iid, _ in recs if iid in mask_index)
    total_test += len(recs); total_hit += hit
    print(f"  {c:6s} {hit:4d} / {len(recs):4d}  ({100 * hit / max(len(recs), 1):5.1f}%)")
print(f"  total  {total_hit} / {total_test}  ({100 * total_hit / total_test:.1f}%)")
assert total_hit > 0, "no test image has a mask, check the mask file names"

print("8) end-to-end check")
model = load_reference_resnet50(RESNET50_CKPT)
_NORM = A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
VAL_TF = A.Compose([A.Resize(COMPARE_SIZE, COMPARE_SIZE), _NORM, ToTensorV2()])

_iid, _path, _cls = next((i, p, c) for i, p, c in test_records_raw if i in mask_index)
_x = VAL_TF(image=np.array(Image.open(_path).convert('RGB')))['image']
_cam, _pred = compute_gradcam(model, _x, None if GRADCAM_TARGET == 'pred' else CLASSES.index(_cls))
_cam = upsample_cam(_cam, COMPARE_SIZE)
_mask = load_mask(mask_index[_iid], COMPARE_SIZE)
print(f"  [ok] {_iid}: true {_cls}, pred {CLASSES[_pred]}, "
      f"EP = {float((_cam * _mask).sum() / (_cam.sum() + 1e-8)):.3f}")

In [ ]:
# 2 · Audit over every test image with a mask, then the two summary figures.
# Resumable: progress saved to Drive every CHECKPOINT_EVERY images.

import time, sys
import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42   # TrueType, Type 3 fonts break in pdflatex
plt.rcParams['ps.fonttype'] = 42

RESUME = True
CHECKPOINT_EVERY = 100

os.makedirs(OUT_DIR, exist_ok=True)
ckpt_path = os.path.join(OUT_DIR, 'gradcam_segmentation_CHECKPOINT.json')

records, errors = [], []
already_done_ids = set()
if RESUME and os.path.isfile(ckpt_path):
    with open(ckpt_path) as f:
        ckpt = json.load(f)
    records = ckpt.get('records', [])
    errors = [tuple(e) for e in ckpt.get('errors', [])]
    already_done_ids = {r['id'] for r in records} | {e[0] for e in errors}
    print(f"[resume] {len(records)} done, {len(errors)} previous errors")

total_n = len(test_records_raw)
pending = [(iid, path, cls) for iid, path, cls in test_records_raw if iid not in already_done_ids]
t0 = time.time()
print(f"[run] {len(pending)} pending of {total_n}")


def save_checkpoint():
    with open(ckpt_path, 'w') as f:
        json.dump({'records': records, 'errors': errors}, f)


for i, (iid, path, cls) in enumerate(pending, start=1):
    if i % CHECKPOINT_EVERY == 0 or i == len(pending):
        print(f"  {i}/{len(pending)}  |  {time.time() - t0:.0f}s")
        save_checkpoint()
        sys.stdout.flush()

    mpath = mask_index.get(iid)
    if mpath is None:
        continue
    try:
        img = np.array(Image.open(path).convert('RGB'))
        x = VAL_TF(image=img)['image']
        true_idx = CLASSES.index(cls)
        target = true_idx if GRADCAM_TARGET == 'true' else None
        cam, pred_idx = compute_gradcam(model, x, target)
        cam = upsample_cam(cam, COMPARE_SIZE)
        mask = load_mask(mpath, COMPARE_SIZE)

        if mask.sum() == 0:
            errors.append((iid, "empty mask"))
            continue

        eps = 1e-8
        energy_prop = float((cam * mask).sum() / (cam.sum() + eps))
        peak = np.unravel_index(int(cam.argmax()), cam.shape)
        pointing_hit = bool(mask[peak])
        camb = cam >= IOU_THR
        inter = np.logical_and(camb, mask).sum()
        union = np.logical_or(camb, mask).sum()
        iou = float(inter / (union + eps)) if union > 0 else float('nan')

        records.append({
            'id': iid, 'true': cls, 'pred': CLASSES[pred_idx],
            'correct': bool(pred_idx == true_idx),
            'energy_prop': energy_prop, 'pointing_hit': pointing_hit,
            'iou': iou, 'mask_frac': float(mask.mean()),
        })
    except Exception as e:
        errors.append((iid, str(e)))

save_checkpoint()

# counted over the full test list so the number is right after a resume
skipped_total = sum(1 for iid, _, _ in test_records_raw if mask_index.get(iid) is None)
print(f"[run] audited {len(records)} | no mask {skipped_total} | errors {len(errors)}")
for iid, msg in errors[:10]:
    print(f"    {iid}: {msg}")
assert records, "nothing audited"


def agg(subset):
    if not subset:
        return None
    ep = np.array([r['energy_prop'] for r in subset])
    iou = np.array([r['iou'] for r in subset], dtype=float)
    hit = np.array([r['pointing_hit'] for r in subset], dtype=float)
    return {
        'n': len(subset),
        'energy_prop_mean': float(np.nanmean(ep)),
        'energy_prop_std': float(np.nanstd(ep)),
        'energy_prop_median': float(np.nanmedian(ep)),
        'pointing_hit_rate': float(np.nanmean(hit)),
        'iou_mean': float(np.nanmean(iou)),
    }


per_class = {c: agg([r for r in records if r['true'] == c]) for c in CLASSES}
subgroups = {
    'malignant': agg([r for r in records if r['true'] in MALIGNANT]),
    'benign': agg([r for r in records if r['true'] not in MALIGNANT]),
    'overall': agg(records),
}
correctness = {
    'correct': agg([r for r in records if r['correct']]),
    'incorrect': agg([r for r in records if not r['correct']]),
}

out = {
    'config': {
        'model': 'resnet50_seed42', 'gradcam_target': GRADCAM_TARGET,
        'compare_size': COMPARE_SIZE, 'iou_threshold': IOU_THR,
        'mask_source': 'tschandl/ham10000-lesion-segmentations',
    },
    'coverage': {
        'images_with_mask': len(records), 'skipped_no_mask': skipped_total,
        'errors': errors,
        'per_class_n': {c: (per_class[c]['n'] if per_class[c] else 0) for c in CLASSES},
    },
    'per_class': per_class,
    'subgroups': subgroups,
    'correctness': correctness,
    'records': records,
}
json_path = os.path.join(OUT_DIR, 'gradcam_segmentation_overlap.json')
with open(json_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f"[ok] {json_path}")
for c in CLASSES:
    pc = per_class[c]
    if pc:
        print(f"  {c:6s} n={pc['n']:4d}  EP={pc['energy_prop_mean']:.3f}  "
              f"PG={pc['pointing_hit_rate']:.3f}  IoU={pc['iou_mean']:.3f}")
print(f"  malignant EP={subgroups['malignant']['energy_prop_mean']:.3f}  "
      f"benign EP={subgroups['benign']['energy_prop_mean']:.3f}  "
      f"overall EP={subgroups['overall']['energy_prop_mean']:.3f}")
print(f"  correct   n={correctness['correct']['n']:4d}  EP={correctness['correct']['energy_prop_mean']:.3f}")
print(f"  incorrect n={correctness['incorrect']['n']:4d}  EP={correctness['incorrect']['energy_prop_mean']:.3f}")

# EP by class
data, labels, colors = [], [], []
for c in CLASSES:
    vals = [r['energy_prop'] for r in records if r['true'] == c]
    if vals:
        data.append(vals); labels.append(c)
        colors.append('#c0392b' if c in MALIGNANT else '#2c7fb8')
fig, ax = plt.subplots(figsize=(7, 4))
bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, showfliers=False)
for patch, col in zip(bp['boxes'], colors):
    patch.set_facecolor(col); patch.set_alpha(0.6)
ax.axhline(np.nanmean([r['energy_prop'] for r in records]), ls='--', lw=1, color='gray')   # overall mean
ax.set_ylabel('Energy proportion inside lesion', fontsize=13)
ax.set_ylim(0, 1)
ax.tick_params(axis='x', labelsize=15)
ax.tick_params(axis='y', labelsize=13)
ax.set_title('Grad-CAM vs segmentation (red: malignant, blue: benign)', fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_gradcam_seg_energy.pdf'))
plt.close(fig)


def draw_row(axes_row, rid, row_label, target):
    p = path_by_id[rid]
    img = np.array(Image.open(p).convert('RGB').resize((COMPARE_SIZE, COMPARE_SIZE)))
    x = VAL_TF(image=np.array(Image.open(p).convert('RGB')))['image']
    cam, _ = compute_gradcam(model, x, target)
    cam = upsample_cam(cam, COMPARE_SIZE)
    mask = load_mask(mask_index[rid], COMPARE_SIZE)
    hm = (plt.get_cmap('jet')(cam)[:, :, :3] * 255).astype(np.uint8)
    overlay = (0.6 * img + 0.4 * hm).clip(0, 255).astype(np.uint8)
    axes_row[0].imshow(img); axes_row[0].set_ylabel(row_label, fontsize=14)
    axes_row[1].imshow(overlay)
    axes_row[2].imshow(img)
    axes_row[2].contour(mask, levels=[0.5], colors='lime', linewidths=1.4)   # expert mask
    axes_row[2].contour(cam, levels=[IOU_THR], colors='red', linewidths=1.0)  # Grad-CAM
    for ax_ in axes_row:
        ax_.set_xticks([]); ax_.set_yticks([])


def set_col_titles(axes):
    for j, t in enumerate(('Original', 'Grad-CAM', 'Lesion vs Grad-CAM')):
        axes[0, j].set_title(t, fontsize=14)


# one correct case per class: first correct record in sorted-id order
path_by_id = {iid: path for iid, path, _ in test_records_raw}
picks = {}
for r in records:
    if r['true'] not in picks and r['correct']:
        picks[r['true']] = r
rows = [c for c in CLASSES if c in picks]
fig, axes = plt.subplots(len(rows), 3, figsize=(7.5, 2.4 * len(rows)), squeeze=False)
for i, c in enumerate(rows):
    draw_row(axes[i], picks[c]['id'], c, None if GRADCAM_TARGET == 'pred' else CLASSES.index(c))
set_col_titles(axes)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig_gradcam_seg_panel.pdf'))
plt.close(fig)
print(f"[ok] figures in {OUT_DIR}")

In [ ]:
# 3 · Misclassified cases whose attention still falls on the lesion.
# Needs cells 1-2 in the same session. Only redraws the selected cases.

N_CASES = 2
MASK_FRAC_MIN, MASK_FRAC_MAX = 0.15, 0.60   # mid-size lesions, so both contours stay readable

error_records = [r for r in records if not r['correct']]
assert error_records, "no misclassified records"

candidates = [r for r in error_records if MASK_FRAC_MIN <= r.get('mask_frac', 0) <= MASK_FRAC_MAX]
if len(candidates) < N_CASES:
    print(f"[warn] only {len(candidates)} mid-size errors, dropping the size filter")
    candidates = error_records

# highest EP first, one case per (true, pred) pair
candidates_sorted = sorted(candidates, key=lambda r: r['energy_prop'], reverse=True)
error_picks, seen_pairs = [], set()
for r in candidates_sorted:
    pair = (r['true'], r['pred'])
    if pair in seen_pairs:
        continue
    error_picks.append(r); seen_pairs.add(pair)
    if len(error_picks) == N_CASES:
        break
for r in candidates_sorted:   # fill up if there are not enough distinct pairs
    if len(error_picks) == N_CASES:
        break
    if r not in error_picks:
        error_picks.append(r)

for r in error_picks:
    print(f"  {r['id']}: true={r['true']} pred={r['pred']} EP={r['energy_prop']:.3f} "
          f"mask_frac={r.get('mask_frac', 0):.2f} pointing={r['pointing_hit']}")

fig, axes = plt.subplots(len(error_picks), 3, figsize=(7.5, 2.4 * len(error_picks)), squeeze=False)
for i, r in enumerate(error_picks):
    draw_row(axes[i], r['id'], f"{r['true']}\n(pred: {r['pred']})", None)   # CAM on the predicted class
set_col_titles(axes)
fig.tight_layout()
out_fig = os.path.join(OUT_DIR, 'fig_gradcam_seg_errors.pdf')
fig.savefig(out_fig)
plt.close(fig)
print(f"[ok] {out_fig}")

In [ ]:
# 4 · EP on correct vs misclassified images: pooled, per class, and against lesion size.
# Standalone: reads only the audit JSON (CPU, no data needed).
# The pooled comparison mixes classes with very different EP and error rates; the per-class
# tests are the ones to interpret.

import os, json
import numpy as np
from scipy.stats import mannwhitneyu, spearmanr

CANDIDATES = ["../results/attention/gradcam_segmentation_overlap.json",   # from notebooks/
              "results/attention/gradcam_segmentation_overlap.json",      # from repo root
              "/content/drive/MyDrive/ISIC2018_Task3/colab/gradcam_segmentation_overlap.json"]
AUDIT_JSON = next(p for p in CANDIDATES if os.path.isfile(p))
CLASSES = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
MALIGNANT = {'MEL', 'BCC', 'AKIEC'}

with open(AUDIT_JSON) as f:
    recs = json.load(f)['records']


def ep_of(rs):
    return np.array([r['energy_prop'] for r in rs])


# pooled (expected with the committed JSON: U = 173609, p = 0.0004)
ok, err = ep_of([r for r in recs if r['correct']]), ep_of([r for r in recs if not r['correct']])
u, p = mannwhitneyu(ok, err, alternative='two-sided')
print(f"pooled    correct n={len(ok):4d} EP={ok.mean():.3f} | incorrect n={len(err):3d} EP={err.mean():.3f} | "
      f"U={u:.0f} p={p:.4f}")

# per class; Bonferroni over the classes with at least 2 errors
per_class = []
for c in CLASSES:
    a = ep_of([r for r in recs if r['true'] == c and r['correct']])
    b = ep_of([r for r in recs if r['true'] == c and not r['correct']])
    u, p = mannwhitneyu(a, b, alternative='two-sided') if len(b) >= 2 else (None, None)
    per_class.append((c, a, b, u, p))
alpha = 0.05 / sum(1 for *_, p in per_class if p is not None)
for c, a, b, u, p in per_class:
    head = f"{c:6s}    correct n={len(a):4d} EP={a.mean():.3f} | incorrect n={len(b):3d}"
    if p is None:
        print(f"{head} (not tested)")
    else:
        print(f"{head} EP={b.mean():.3f} | U={u:.0f} p={p:.2g} {'*' if p < alpha else ''}")
print(f"* significant after Bonferroni (alpha = {alpha:.4f})")

# lesion size: EP depends strongly on the mask fraction, so compare lift = EP / mask fraction as well
ep_all = ep_of(recs)
mf = np.array([r['mask_frac'] for r in recs])
rho, _ = spearmanr(ep_all, mf)
print(f"\nEP vs mask fraction: Spearman rho = {rho:.2f}")
lift_mal = np.array([r['energy_prop'] / r['mask_frac'] for r in recs if r['true'] in MALIGNANT])
lift_ben = np.array([r['energy_prop'] / r['mask_frac'] for r in recs if r['true'] not in MALIGNANT])
u, p = mannwhitneyu(lift_mal, lift_ben, alternative='two-sided')
print(f"lift (EP / mask fraction): malignant {lift_mal.mean():.2f} vs benign {lift_ben.mean():.2f}, "
      f"U={u:.0f} p={p:.2g}")